# M32 · HYPOINVERSE through PocketQuake: a production locator, its files and its residuals

**Part 2 · week 4 · about 7.5 h core**

## 한국어 요약

이 모듈에서는 연구실 처리 도구 PocketQuake 안에서 미국지질조사소(USGS)의 진원 결정 프로그램 HYPOINVERSE를 sgtl 도구로 실행하고, 그 결과 파일을 직접 읽어 해석합니다. 먼저 실행 기록(runs.jsonl)에서 실행이 남긴 파일의 SHA-256 지문을 찾아 지금 디스크에 있는 파일과 대조하고, 관측소별 잔차와 가중치가 담긴 아카이브 파일(.arc)을 고정 폭 형식 그대로 읽는 함수를 작성해 요약 파일(.sum)의 값과 비교합니다. 이어서 2018-2022년 상주 지진 6개의 HYPOINVERSE 진원을 기상청 목록, 그리고 여러분이 M31에서 직접 구한 진원과 짝지어 진앙 거리, 깊이 차이, 발생 시각 차이를 계산합니다. 아무것도 바꾸지 않은 재실행이 같은 파일을 내는지 확인하고, 속도 구조 모델을 kim1983에서 kim2011로 바꾸었을 때 진원이 어떻게 달라지는지 설명합니다. 마지막으로 관측소마다 잔차의 평균을 구해 일관되게 이르거나 늦은 관측소를 찾고, 관측소 보정값(station delay)이 무엇을 담을 수 있고 무엇을 담을 수 없는지 생각합니다.

## What you will be able to do

- Run PocketQuake's HYPOINVERSE stage through sgtl (a dry run first) and use the SHA-256 fingerprints of the run log to show which files a run wrote
- Read HYPOINVERSE's archive file by its fixed columns and reproduce, for every earthquake, the number of weighted readings and the RMS residual of its summary file
- Match the HYPOINVERSE locations with the KMA catalog and with your own location of module M31, and report epicentral distances, depth differences and origin-time differences
- Rerun the location unchanged, compare the files line by line, and explain how the locations change from velocity model kim1983 to kim2011
- Find the stations whose residuals are consistently early or late, and explain what a station delay can and cannot hold

## Before you start

- Modules: M21, M30, M31
- Reading: Klein (2002), the sections on weighting (the DIS, RMS and WET commands), on the homogeneous layer model (CRH) and on the Y2000 archive format; Husen & Hardebeck (2010), the sections on what controls location accuracy (picks, station geometry, velocity model); Kissling, Ellsworth, Eberhart-Phillips & Kradolfer (1994), the parts on station corrections (a skim is enough)
- Needs: stp, pocketquake
- Key terms (see the glossary): HYPOINVERSE (진원 결정 프로그램), PocketQuake (지진 분석 도구 모음), cluster (PocketQuake; 클러스터 (PocketQuake)), absolute location (절대 진원 결정), Geiger's method (가이거 방법), dry run (드라이 런, 시험 실행), checksum (SHA-256 fingerprint; 체크섬), JSON Lines (JSON 줄 형식), deterministic computation (결정론적 계산), summary file (HYPOINVERSE; 요약 파일 (HYPOINVERSE)), archive file (HYPOINVERSE; 아카이브 파일 (HYPOINVERSE)), fixed-width format (고정 폭 형식), implied decimal point (생략된 소수점), weight code (가중치 코드), PhaseNet+ (PhaseNet+), pick probability (피킹 확률), distance weighting (거리 가중), residual weighting (잔차 가중), trial depth (시작 깊이), first arrival (최초 도달파 (초동)), head wave (선두파), crossover distance (교차 거리), Pn and Sn (Pn파와 Sn파), velocity model (속도 구조 모델), residual (잔차), RMS residual (RMS 잔차), location errors (ERH and ERZ; 위치 오차 (ERH와 ERZ)), error ellipse (오차 타원), azimuthal gap (방위각 공백), distance to the nearest station (DMIN; 최근접 관측소 거리 (DMIN)), location quality (진원 결정 품질 등급), catalog matching (지진 목록 대조), aware datetime (인식 시각, 시간대 정보가 있는 시각), time zone (시간대), UTC (협정 세계시), KST (한국 표준시), origin time (진원시, 발생 시각), epicenter (진앙), epicentral distance (진앙 거리), focal depth (진원 깊이), station delay (station correction; 관측소 보정값), standard error (표준 오차), phase misidentification (위상 오인), S-P time (S-P 시간), Vp/Vs ratio (Vp/Vs 비)

## Passing this module

Each task is done when its `st.check(...)` cell prints PASS. The pass criteria are frozen in [criteria/M32.yaml](../../criteria/M32.yaml).

## AI use

Follow [docs/ai_policy.md](../../docs/ai_policy.md) and log every AI conversation in `work/ai_log/M32.md`.

---

© 2026 Minseong Seo (SGTL, Seoul National University) and contributors · code MIT · text CC BY 4.0 · cite CITATION.cff

In [ ]:
# Parameters: the only cell with values you may change. Run it first, then the setup cell.
DATASET = "sangju2019"                    # folder of your STP download (module M03) inside the data folder
PQ_SLUG = "sangju_m21"                    # your PocketQuake cluster of module M21 (staged and picked there)
PQ_CORES = 1                              # cores of each PocketQuake run (at most 4 on the shared server)
PQ_MEM_GB = 8.0                           # memory cap of each run in GB (its picker needs about 3.5)
MAX_DT_S = 5.0                            # two locations of one earthquake: origin times at most this far apart (s)
M31_FILE = "../M31_location/results/M31_hypoinverse_runs.csv"   # your location table of module M31 (task M31-T6)
M31_EVENT_ID = "2019009762"               # the earthquake that table locates (the ML 3.9 mainshock; KMA id)
MIN_EVENTS = 4                            # task 5: a station term needs at least this many earthquakes
FIGURE_DIR = "figures"                    # your figure files go here, next to this notebook
RESULT_DIR = "results"                    # your result files go here, next to this notebook

In [ ]:
import sgtl_tutorial as st
S = st.init("M32")

import hashlib
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Path(FIGURE_DIR).mkdir(exist_ok=True)
Path(RESULT_DIR).mkdir(exist_ok=True)
RUN_DIR = st.pq.pipeline_dir() / "pipeline" / "runs" / PQ_SLUG      # PocketQuake's results for your cluster
HYP_DIR = RUN_DIR / "1.HypoInv"                                     # what its location stage writes
print("PocketQuake results of your cluster:", RUN_DIR)

## How this notebook works

The same five steps as in the earlier modules, for every task: read the **Contract**, write a **prediction**,
replace `raise NotImplementedError(...)` under `# YOUR CODE HERE` with your code, run the check cell, and answer
the **Explain** prompt in 3-5 sentences of your own. `st.hint("M32-T1", 1)` shows a first hint; hints 2 and 3
unlock after failed checks. The checks try your functions on made-up files and tables whose answers are known from
how they were built (`st.reroll("M32-T1")` draws new ones), and then on products of your own PocketQuake runs; for
those they never print the values they compare with.

This module needs three things from earlier work: your PocketQuake cluster of module M21 (`PQ_SLUG`, staged and
picked there), the course tools (`bin/sgtl-install tools`, which put PocketQuake, EQNet and HYPOINVERSE under
`~/sgtl`), and your location table of module M31 (`work/M31_location/results/M31_hypoinverse_runs.csv`, needed
from Part 4 on). Nothing here downloads anything. The location runs take a second; the picking run of Part 2 takes
one to two minutes on one core. On the shared server use at most 4 cores for any run (`PQ_CORES`), and one heavy
run at a time.

You may ask an AI tool to explain an error message or a concept. It may not write the task functions, your
predictions or your Explain answers ([docs/ai_policy.md](../../docs/ai_policy.md)); log AI conversations in
`work/ai_log/M32.md`.

## Part 0 · Your cluster and the six earthquakes (about 15 min)

Your STP download `sangju2019` holds six earthquakes near Sangju (2018-2022). The cell lists them from the KMA
catalog that ships with the course (times in UTC; KMA, reference key `kma_catalog`) and checks that PocketQuake
has your cluster of module M21 with its picks. Your STP folder names each earthquake by its UTC **origin time**
(glossary `origin_time`), so the names find the catalog rows.

In [ ]:
if not (st.pq.pipeline_dir() / "pipeline" / "clusters" / f"{PQ_SLUG}.py").is_file() or \
        not list((RUN_DIR / "picks").glob("*_picks.csv")):
    raise FileNotFoundError(f"PocketQuake has no picked cluster {PQ_SLUG!r} under {RUN_DIR.parent}. This module "
                            "locates the picks of your PocketQuake cluster of module M21: finish task M21-T4 of module "
                            "M21 first (or set PQ_SLUG to the cluster you used there).")
SAC_ROOT = st.data.root() / DATASET / "stp_download" / "SAC"
EVENT_FOLDERS = sorted(p.name for p in SAC_ROOT.iterdir() if p.is_dir())
kma = st.data.sequence("sangju2019")
names = kma["origin_utc"].dt.strftime("%Y%m%d%H%M%S")
six = kma[names.isin(EVENT_FOLDERS)].reset_index(drop=True)
print(f"{len(EVENT_FOLDERS)} earthquakes in your download; {len(six)} of them in the KMA catalog subset "
      f"'sangju2019' ({len(kma)} events within {kma.attrs['sequence']['radius_km']} km, 2018-2022):")
print(six[["event_id", "origin_utc", "latitude", "longitude", "depth_km", "ml"]].to_string(index=False))
print(f"\nyour cluster {PQ_SLUG!r}: {len(list((RUN_DIR / 'picks').glob('*_picks.csv')))} pick files")

KMA reports epicenters to 0.01° and depths to whole kilometers. Keep that in mind when you compare locations: a
difference smaller than the rounding means nothing.

## Part 1 · What HYPOINVERSE does (about 30 min)

**HYPOINVERSE** (glossary `hypoinverse`; Klein 2002) is the U.S. Geological Survey's program for locating
earthquakes. It solves the problem of module M31, an **absolute location** (glossary `absolute_location`), with
**Geiger's method** (glossary `geiger_method`): start from a trial hypocenter, fit the travel-time residuals with
small changes of the epicenter, depth and origin time, and repeat. A production tool adds the details that real
data need. PocketQuake runs version 1.40 (`hyp1.40`); the course installer (`bin/sgtl-install tools`) compiled it
on your server from the USGS source code, so it is the program the U.S. networks use, not a copy written for the
course. At the end of M31 you ran it yourself on your own picks of one earthquake; here PocketQuake writes its
input files from its own picks and runs it for every earthquake of your cluster.

What PocketQuake's location stage does (the next cell prints the lines from your copy):

1. **Phases.** It writes a phase file (`1.HypoInv/PHS/<Region>.phs`) from the P and S picks of its picking stage
   (PhaseNet+, glossary `phasenet_plus`, module M21).
2. **Weight codes.** Every pick gets a **weight code** (glossary `pick_weight_code`) from 0 (full weight) to 4 (no
   weight). PocketQuake sets it from the **pick probability** (glossary `pick_probability`) of PhaseNet+: code 0
   from 0.9 up, 1 from 0.7, 2 from 0.5, 3 from 0.3, 4 below. HYPOINVERSE turns codes 0, 1, 2 and 3 into the
   weights 1, 0.75, 0.5 and 0.25 (its default, the WET command; Klein 2002).
3. **Two velocity models**, kim1983 and kim2011 (module M30), as layers of constant speed for P and for S
   (HYPOINVERSE's homogeneous layer model, the CRH command). Its travel-time routine comes from HYPO71 (Lee & Lahr
   1975): for each station it computes the direct wave and the head wave along every boundary below the source, and
   keeps the earliest. So HYPOINVERSE predicts the **first arrival** (glossary `first_arrival`), the onset a picker
   marks. In this model every station sits on the top surface: station heights are not used.
4. **Control commands** (the `HypControl` line):
   - `ZTR 10 F`: start every earthquake at 10 km, the **trial depth** (glossary `trial_depth`), and let the depth
     move (F: do not fix it).
   - `DIS 4 50 1 3`, the **distance weighting** (glossary `distance_weighting`): from iteration 4 on, stations
     closer than D keep full weight, stations beyond 3 × D get none, with a smooth (cosine) taper between. D is the
     distance to the second-closest station, but never less than 50 km. PocketQuake offers only stations within
     100 km of the epicenter.
   - `RMS 4 0.12 2 4`, the **residual weighting** (glossary `residual_weighting`): from iteration 4 on, a reading
     whose residual is larger than 2 × the event's RMS residual starts to lose weight, and one larger than 4 × the
     RMS gets none (an RMS below 0.12 s counts as 0.12 s). A single wrong pick cannot pull the solution far.
   - `MIN 4`: an earthquake recorded at fewer than 4 stations is not located; `CON 50 ...`: at most 50 iterations.
   - `LET 5 2 0 0 0`: a station is known by its full five-letter code and its network, so two networks' stations
     with the same code stay apart.
5. **One run, both models.** The stage locates every earthquake once per model and writes, in
   `1.HypoInv/<model>/`, a **summary file** `<Region>.sum` (glossary `summary_file`: one line per earthquake), an
   **archive file** `<Region>.arc` (glossary `archive_file`: per earthquake a header line, one line per station
   with the residuals and weights HYPOINVERSE used, and a closing line) and a print file `<Region>.prt` for
   people. `st.pq.read_sum(PQ_SLUG, "kim1983")` reads one summary file.

Why the travel-time physics of M30 matters here: with stations out to 100 km, the first arrival at the far stations
can be a head wave (glossary `head_wave`) along a boundary in the crust; in kim1983 the boundary at 15 km lies just
below these sources, so it overtakes the direct wave beyond the **crossover distance** (glossary
`crossover_distance`) you computed in M30-T2 (a few tens of kilometers). HYPOINVERSE predicts that wave, so a far
pick is compared with the right arrival. Head waves along the Moho (**Pn** and **Sn**, glossary `pn_sn`) come first
only beyond about 120-135 km for sources near 14 km in both models (your functions of M30-T2 give the exact
distances), so with stations within 100 km they never take part: the lab rule (locate with direct P and S within
about 100 km, never with Pn or Sn) holds here.

In [ ]:
PIPE = st.pq.pipeline_dir() / "pipeline"


def show_lines(path, words, last=None):
    """Print the lines of a file of your PocketQuake copy that contain any of the words (with line numbers)."""
    lines = Path(path).read_text(encoding="utf-8").splitlines()
    for number, line in enumerate(lines[:last], 1):
        if any(w in line for w in words):
            print(f"{Path(path).name}:{number}: {line}")


show_lines(PIPE / "clusters" / "_base.py", ["STD_HYP =", "RMS=(4", "KIM1983 =", "KIM2011 =", "=((", "s=(("])
show_lines(PIPE / "config.py", ["(0.90, 0)", "(0.50, 2)", "LET: tuple"])
print()
show_lines(PIPE / "core" / "hypoinverse.py", ["CRH 1", "CRH 2", "DIS {DIS}", "RMS {RMS}", "ZTR {ZTR0}",
                                              "SUM '", "ARC '"])
print("\nYour cluster module (made by st.pq.stage in M21):")
print((PIPE / "clusters" / f"{PQ_SLUG}.py").read_text(encoding="utf-8").split('"""', 2)[-1].strip())

The summary file has one line per earthquake. `st.pq.read_sum` gives these columns: `id` (PocketQuake's number of
the earthquake), `time` (origin time, UTC), `lat`, `lon`, `depth` (km), `num` (the readings whose final weight is
above 0.1), `gap`, `rms`, `erh`, `erz` and `qual`:

- `gap` is the **azimuthal gap** (glossary `azimuthal_gap`) in degrees;
- `rms` is HYPOINVERSE's **RMS residual** (glossary `rms_residual`), with weights: RMS² = Σ (wᵢ rᵢ)² / Σ wᵢ², where
  rᵢ is the **residual** (glossary `residual`, observed minus predicted time) and wᵢ the final weight of reading i
  (Klein 2002, appendix 2);
- `erh` and `erz` are the **location errors** (glossary `location_errors`) in km: the largest horizontal and
  vertical extents of the **error ellipsoid** (glossary `error_ellipse`) of the final iteration;
- `qual` is a **location quality** letter (glossary `location_quality`), A (best) to D, the average of one rating
  from RMS, ERH and ERZ and one from the station geometry (the number of readings, the gap and the **distance to
  the nearest station**, glossary `nearest_station_distance`, against the depth). Klein (2002) calls it
  oversimplified and often misused: read the numbers it is made from instead.

## Part 2 · The first run and its log (about 60 min, task 1)

Your cluster's picks are those of the last picking run of M21, and that run used a raised threshold
(`PNPLUS_MIN_PROB`). This module locates the picks of PocketQuake's default settings, so the first run repeats the
picking stage (no setting changed) and then locates: stages `picking` to `hypoinverse`.

Every run through `st.pq` appends one line to the run log `work/.sgtl/runs.jsonl` of your repository. The log is
in **JSON Lines** (glossary `json_lines`): one JSON object per line. A line says what ran (`command`, `cluster`,
`stage_from`, `through`), whether it worked (`returncode`, 0 = success), when (`ts`, UTC), and the **SHA-256
fingerprint** (glossary `checksum`) of every product file the run created or changed (`products`: the path inside
the cluster's run folder, and its fingerprint). A fingerprint changes when a single byte of the file changes, so it
tells you whether the file you read now is the one a given run wrote.

### M32-T1 · The run and its log · core · about 60 min

**Goal:** run PocketQuake's picking and location stages through `st.pq` (a dry run first), write
`run_products(log_path, slug, stage, stages)`, which lists what the runs logged, and use it to show that the
location files now in your cluster are the ones your last run wrote.

**Given:** the run cells below and your run log `work/.sgtl/runs.jsonl`.

**Contract (the reader):**
- Input: `log_path` (a `str` or `pathlib.Path`) of a run log; `slug` (a cluster name); `stage` (one stage name);
  `stages` (every stage name in run order, as `st.pq.STAGES`).
- A matching run is a line whose `command` is `"pq run"`, whose `cluster` is `slug`, whose `returncode` is 0, and
  whose stages contain `stage`: `stage_from` comes at or before `stage` in `stages`, and `through` at or after it.
  Lines of other commands (for example `"pq bootstrap"`) have no `stage_from` or `through`.
- Output: a pandas DataFrame with the columns `run`, `ts`, `product` and `sha256`, one row per product of every
  matching run, in log order (the products of one run in the order the line lists them). `run` numbers the
  matching runs 0, 1, 2, ... in log order; a matching run that logged no products gets its number but no rows.
  With no matching run, give an empty DataFrame with these four columns.
- A line that is empty or is not valid JSON is skipped (a run stopped while writing can leave half a line).
- Example (made up): a log with a matching run of cluster `qz_demo` at `2031-03-17T12:46:00Z` that logged one file
  gives one row: `run` 0, `ts` `"2031-03-17T12:46:00Z"`, `product` (the logged path) and `sha256` (64 hex
  letters).

**Contract (the report):** `first_run = {"reader": run_products, "slug": PQ_SLUG, "fingerprints": {<path>:
<sha256>}}`, one entry for every .sum, .arc and .prt file now in the model folders of your cluster's 1.HypoInv
folder: `<path>` is the file's path as your run log writes paths (for example `"1.HypoInv/kim1983/<Region>.sum"`)
and `<sha256>` its SHA-256 fingerprint now (64 hex letters), computed with `hashlib` from the file's bytes.

**Done when** `st.check("M32-T1", answer=first_run)` passes: the check runs your reader on 8 made-up logs (failed
runs, other clusters, other stages, other commands, empty and broken lines), then on your own log; then it
computes the fingerprints of your current .sum, .arc and .prt files itself: yours must be those, one for every
file, and for the files the log lists they must be the ones your last location run logged (the files are the ones
that run wrote). It never prints them.

**Stuck?** `st.hint("M32-T1", 1)`

In [ ]:
# Before you run: this run repeats M21's picking with the default settings. Will its picks be the same, byte for
# byte, as those of your first (default) run in M21, and how could you tell from the log alone? Which files will the
# log list for the location stage, and what could make a file's fingerprint differ from the logged one later on?
st.predict("M32-T1", """

""")

In [ ]:
def run_products(log_path, slug, stage, stages):
    """One row per product of every successful 'pq run' of the cluster whose stages include the stage.

    log_path : str or pathlib.Path of a run log in JSON Lines
    slug     : the cluster name;  stage : one stage name;  stages : all stage names in run order
    returns  : pandas DataFrame with the columns run, ts, product, sha256 (in log order)
    """
    # Step 1: read the log line by line; skip a line that is empty or not valid JSON.
    # Step 2: keep the successful 'pq run' lines of this cluster whose range of stages holds the stage.
    # Step 3: number those runs 0, 1, 2, ... and add one row per product they logged.
    # YOUR CODE HERE
    raise NotImplementedError("M32-T1: write your code here")

The run: read the dry run (glossary `dry_run`) first. It prints the command, the cores, the memory cap and where
the results go, and starts nothing.

In [ ]:
st.pq.run(PQ_SLUG, "picking", "hypoinverse", cores=PQ_CORES, mem_gb=PQ_MEM_GB)          # dry run

In [ ]:
assert st.pq.run(PQ_SLUG, "picking", "hypoinverse", cores=PQ_CORES, mem_gb=PQ_MEM_GB, apply=True) == 0
LOG = st.data.repo_path("work", ".sgtl", "runs.jsonl")

In [ ]:
# Fill in first_run. For fingerprints, find every .sum, .arc and .prt file in the model folders of HYP_DIR and
# compute its SHA-256 now (hashlib.sha256(path.read_bytes()).hexdigest()); key it by its path as the log writes
# paths, which start with "1.HypoInv/".
# YOUR CODE HERE
raise NotImplementedError("M32-T1: write your code here")
print(f"reader {first_run['reader'].__name__}, slug {first_run['slug']!r}, "
      f"{len(first_run['fingerprints'])} fingerprints")

In [ ]:
st.check("M32-T1", answer=first_run)

What your last run logged, where its picks fingerprint appears in your log, and how many of your fingerprints are
of files the log lists and equal to what your last location run logged:

In [ ]:
everything = run_products(LOG, PQ_SLUG, "picking", st.pq.STAGES)
last = everything[everything["run"] == everything["run"].max()]
print(last[["ts", "product", "sha256"]].assign(sha256=last["sha256"].str[:16] + "...").to_string(index=False))
picks_now = last.loc[last["product"].str.startswith("picks/"), "sha256"]
if len(picks_now):
    same = everything[(everything["sha256"] == picks_now.iloc[0]) & (everything["run"] < everything["run"].max())]
    print(f"\nthe picks fingerprint of this run appears in {same['run'].nunique()} earlier picking run(s) of "
          f"{PQ_SLUG!r}, logged at: {', '.join(same['ts'])}")
located = run_products(LOG, PQ_SLUG, "hypoinverse", st.pq.STAGES)
newest = located[located["run"] == located["run"].max()].set_index("product")["sha256"]
listed = [path for path in first_run["fingerprints"] if path in newest.index]
agree = sum(newest[path] == first_run["fingerprints"][path] for path in listed)
print(f"\nyour fingerprints: {len(first_run['fingerprints'])}; of files the last location run logged: "
      f"{len(listed)}, equal to the logged value: {agree}; of files it did not log: "
      f"{len(first_run['fingerprints']) - len(listed)}")

**Explain (3-5 sentences):** Were the picks of this run the same as those of a default picking run of M21, and how
does the log show it without opening a pick file? Which files did the location stage log, and which files that it
writes are not in the log? Why does a fingerprint tell you more than a file's date, and when would a file's
fingerprint stop matching the one your last run logged?

## Part 3 · The archive file, column by column (about 90 min, task 2)

The summary file gives one line per earthquake; the archive file shows how HYPOINVERSE got there, station by
station. It is a **fixed-width format** (glossary `fixed_width_format`): every value sits in fixed columns, and
neighboring values often touch with no blank between them. Numbers carry no decimal point; the format says where
it goes, an **implied decimal point** (glossary `implied_decimal_point`). Klein (2002, "Y2000 (station) archive
format") lists the columns. Those you need, counted from 1 as in the manual:

| Columns | Format | Value |
|---|---|---|
| 1-5 | A5 | station code (left-justified) |
| 6-7 | A2 | network code |
| 14-15 | A2 | P remark such as `IP`; blank when the line holds no P reading |
| 35-38 | F4.2 | P travel-time residual (s) |
| 39-41 | F3.2 | P weight actually used |
| 47-48 | A2 | S remark such as `ES`; blank when the line holds no S reading |
| 51-54 | F4.2 | S travel-time residual (s) |
| 64-66 | F3.2 | S weight actually used |
| 75-78 | F4.1 | epicentral distance (km) |
| 92-94 | F3.0 | azimuth from the earthquake to the station (degrees east of north) |

F4.2 means 4 columns whose last two digits come after the implied decimal point: `  -7` is −0.07 and `1234` would
be 12.34. F4.1 has one implied decimal, F3.0 none; a blank field is 0. In Python, columns a to b (counted from 1)
are `line[a - 1:b]`. Two made-up station lines with a column ruler (station QQA1 of network XX is invented):

```
         1         2         3         4         5         6         7         8         9
1234567890123456789012345678901234567890123456789012345678901234567890123456789012345678901234
QQA1 XX  HHZ IPD1203103171246  605  -7124    0       0      0 0  0   0   0 23410600  0     211
QQA1 XX  HHZ    0203103171246    0   0  0 1895ES 2  31      0 0 62   0   0 23416100  0     211
```

The first line holds a P reading (residual −0.07 s, weight 1.24); the second an S reading (residual 0.31 s,
weight 0.62); both are 23.4 km from the earthquake at an azimuth of 211°. Look at `-7124` and `23410600`: a residual
touches a weight, and a distance touches the next field.

Three kinds of lines: a **header line** for each earthquake starts with its four-digit year in columns 1-4 and
holds the earthquake's ID number in columns 137-146 (the `id` of the summary file); the **station lines** of that
earthquake follow; a **closing line** is blank in columns 1-5 (it holds only the ID number, further right). One
station line may hold a P and an S reading together. Your run writes one line per reading, but your reader must
not rely on that.

### M32-T2 · The archive file, column by column · core · about 90 min

**Goal:** write `read_arc(path)`, which turns a HYPOINVERSE archive file into a table with one row per reading.

**Given:** the column table above, and your archive files `HYP_DIR / "kim1983" / "<Region>.arc"` and
`HYP_DIR / "kim2011" / "<Region>.arc"`.

**Contract:**
- Input: `path` (a `str` or `pathlib.Path`) of an archive file in HYPOINVERSE's Y2000 format.
- Output: a pandas DataFrame with one row per P or S reading and the columns `event` (int, the ID number of the
  header line above the reading), `station` (text `network.station`, such as `XX.QQA1`), `phase` (`"P"` or
  `"S"`), `residual_s`, `weight`, `distance_km` and `azimuth_deg` (floats), in file order; a line that holds both
  readings gives its P row first.
- A line holds a P reading when columns 14-15 are not blank, and an S reading when columns 47-48 are not blank.
  Header lines, closing lines and empty lines give no rows.

**Done when** `st.check("M32-T2", func=read_arc, slug=PQ_SLUG)` passes: 10 made-up archive files (the first with
blanks between all fields, the others with touching fields, negative residuals, distances above 100 km and lines
with both readings), then your two real archive files, compared with HYPOINVERSE's summary files of the same run:
the same earthquakes, and for each earthquake the number of readings with weight above 0.1 (`num`) and the
weighted RMS residual (`rms`), within rounding. It never prints the real values.

**Stuck?** `st.hint("M32-T2", 1)`

In [ ]:
# Before you code: a colleague reads the station lines with line.split(). On which of the two lines above does that
# go wrong, and how? In your run, how many of the about 580 readings per model do you expect to get weight 0, and
# what would make HYPOINVERSE give a pick no weight at all?
st.predict("M32-T2", """

""")

In [ ]:
def read_arc(path):
    """One row per P or S reading of a HYPOINVERSE Y2000 archive file.

    path    : str or pathlib.Path of the .arc file
    returns : pandas DataFrame with event (int), station ('network.station'), phase ('P' or 'S'), residual_s,
              weight, distance_km, azimuth_deg (floats), in file order (a line's P row before its S row)
    """
    # Step 1: decide the kind of each line: header (year in columns 1-4), closing (columns 1-5 blank), station.
    # Step 2: on a header line, remember the ID number of columns 137-146.
    # Step 3: on a station line, read each field from its columns with its implied decimal point (blank = 0), and
    #         add a P row when columns 14-15 are not blank and an S row when columns 47-48 are not blank.
    # YOUR CODE HERE
    raise NotImplementedError("M32-T2: write your code here")

In [ ]:
st.check("M32-T2", func=read_arc, slug=PQ_SLUG)

Your two archive files in numbers, and the readings that HYPOINVERSE gave no weight:

In [ ]:
MODELS = ("kim1983", "kim2011")
ARC = {m: sorted((HYP_DIR / m).glob("*.arc"))[0] for m in MODELS}
phases = {m: read_arc(ARC[m]) for m in MODELS}
for m in MODELS:
    table = phases[m]
    zero = table[table["weight"] == 0]
    print(f"{m}: {len(table)} readings ({(table['phase'] == 'P').sum()} P, {(table['phase'] == 'S').sum()} S) "
          f"at {table['station'].nunique()} stations; weight above 0.1: {(table['weight'] > 0.1).sum()}; "
          f"weight 0: {len(zero)}")
zero83 = phases["kim1983"][phases["kim1983"]["weight"] == 0]
print("\nkim1983, the readings with weight 0:")
print(zero83[["event", "station", "phase", "residual_s", "distance_km"]].to_string(index=False))

**Explain (3-5 sentences):** Why does a reader that splits at blanks fail on these files, and how did your reader
avoid it? Which readings got weight 0, at what distances, and with residuals of which size and sign? An **S-P
time** (glossary `s_minus_p_time`) grows by about one second per 8 km of distance: what could a P residual of
about +10 s at 85-95 km be, and what is it called when a picker labels an arrival with the wrong phase (glossary
`phase_misidentification`)? Which HYPOINVERSE setting kept these readings out of the locations?

## Part 4 · Three locations of each earthquake (about 75 min, task 3)

You now have several locations of the Sangju mainshock: PocketQuake's HYPOINVERSE (PhaseNet+ picks at many
stations), the KMA catalog's, and two of your own from module M31, which it saved in `M31_FILE`: your Geiger location
(row `your geiger`: your M20 picks at a few stations, your travel-time function) and HYPOINVERSE run on those same
picks (row `run 1`). The two HYPOINVERSE locations share the program, the model (kim1983: the same layers in both
runs' model files) and the commands: M31 sent PocketQuake's `HYP_TEMPLATE` line by line with the values of
`STD_HYP`, and only the file names in the lines `STA`, `CRH 1`, `CRH 2`, `PHS`, `PRT`, `SUM` and `ARC` differ (the
template also starts with an empty line, which changes nothing). What differs is the input: the picks (yours from
M20 against PhaseNet+'s), so the stations in the station and phase files, and the weight codes (all 0 in M31, from
the pick probability here). M31's table gives the origin time as seconds after KMA's (`origin - KMA (s)`).

To compare locations of the same earthquake from two lists you first pair them: **catalog matching** (glossary
`catalog_matching`) by origin time. That only works when both times are on the same clock: your STP folder's
`event_catalog.csv` is in **KST** (glossary `kst`, UTC + 9 h), the shipped KMA catalog and HYPOINVERSE's files are
in **UTC** (glossary `utc`). A pandas time that carries its **time zone** (glossary `time_zone`) is an **aware**
time (glossary `aware_datetime`); comparing aware times is safe whatever their zones.

The cell below puts each list in the same form (columns `time`, `lat`, `lon`, `depth`, times aware) without
printing it.

In [ ]:
if not Path(M31_FILE).is_file():
    raise FileNotFoundError(f"{M31_FILE} is missing. Part 4 compares HYPOINVERSE with your own locations of the "
                            "mainshock: finish task M31-T6 of module M31 first (its cells save that table), or set "
                            "M31_FILE in the parameters cell to where you saved it.")


def sum_table(summary):
    """st.pq.read_sum's table with aware UTC times (the form match_locations takes)."""
    out = summary.copy()
    out["time"] = pd.to_datetime([str(t) for t in summary["time"]], utc=True)
    return out


def catalog_table(catalog, time="origin_utc", lat="latitude", lon="longitude", depth="depth_km"):
    """A catalog (such as KMA's) with the columns time (aware), lat, lon, depth."""
    out = catalog.rename(columns={time: "time", lat: "lat", lon: "lon", depth: "depth"}).copy()
    out["time"] = pd.to_datetime(out["time"], utc=True)
    return out


hyp83 = sum_table(st.pq.read_sum(PQ_SLUG, "kim1983"))
kma_tab = catalog_table(kma)
m31_raw = pd.read_csv(M31_FILE)
mine = m31_raw[m31_raw["location"].isin(["your geiger", "run 1"])].reset_index(drop=True)
if mine.empty:
    raise ValueError(f"{M31_FILE} has no rows 'your geiger' or 'run 1'; rerun the cells of M31's last part")
m31_origin = kma.loc[kma["event_id"] == M31_EVENT_ID, "origin_utc"].iloc[0]
m31_tab = pd.DataFrame({"location": mine["location"].replace({"run 1": "HYPOINVERSE on your picks"}),
                        "time": m31_origin + pd.to_timedelta(mine["origin - KMA (s)"], unit="s"),
                        "lat": mine["latitude"], "lon": mine["longitude"], "depth": mine["depth (km)"]})
print(f"tables ready: HYPOINVERSE kim1983 {len(hyp83)} rows, KMA {len(kma_tab)} rows, your M31 table "
      f"{len(m31_tab)} row(s)")

### M32-T3 · Three locations of each earthquake · core · about 75 min

**Goal:** write `match_locations(ref, other, max_dt_s)`, which pairs every earthquake of one list with the same
earthquake in another list and measures how far apart the two locations are.

**Given:** the tables `hyp83` (PocketQuake's HYPOINVERSE, kim1983), `kma_tab` (KMA, 27 events) and `m31_tab` (your
two M31 locations of the mainshock).

**Contract:**
- Input: `ref` and `other`, pandas DataFrames with the columns `time` (aware pandas times, any time zone), `lat`
  and `lon` (degrees) and `depth` (km), and any index; `max_dt_s` (seconds).
- For each row of `ref`, its partner is the row of `other` with the closest origin time, if that time is at most
  `max_dt_s` away; otherwise it has no partner.
- Output: a pandas DataFrame with the index of `ref` (same rows, same order) and the columns `match` (the index
  label of the partner in `other`, or None), `dt_s` (partner's time minus ref's time, s), `dist_km` (the
  **epicentral distance**, glossary `epicentral_distance`, between the two epicenters along the Earth's surface,
  km; a sphere of radius 6371 km is accurate enough) and `ddepth_km` (partner's depth minus ref's depth, km); NaN
  in the three numbers when there is no partner.
- A table whose times carry no time zone cannot be compared with anything: raise a `ValueError` that says so.
- Example (made up): ref at `2031-03-17 03:46:10+00:00` (35.10 N, 127.20 E, 8.0 km) and other at
  `2031-03-17 12:46:11.5+09:00` (35.11 N, 127.20 E, 9.5 km) are a pair with `dt_s` 1.5, `dist_km` about 1.11 and
  `ddepth_km` 1.5.

**Done when** `st.check("M32-T3", func=match_locations, slug=PQ_SLUG)` passes: 12 made-up pairs of tables (other
time zones, decoys, near misses just outside the window, two candidates for one earthquake) and one table without
time zones, then your kim1983 summary against the KMA catalog through your function (every earthquake must find its
KMA partner within loose bounds; no values are printed).

**Stuck?** `st.hint("M32-T3", 1)`

In [ ]:
# Before you code: how far apart do you expect HYPOINVERSE's (kim1983) and KMA's epicenters to be, and their
# depths? KMA rounds epicenters to 0.01 degree and depths to whole kilometers: from what size on does a difference
# mean something? Which of your two M31 locations should agree best with PocketQuake's, and why?
st.predict("M32-T3", """

""")

In [ ]:
def match_locations(ref, other, max_dt_s):
    """Pair each earthquake of ref with the one of other closest in origin time (within max_dt_s).

    ref, other : DataFrames with time (aware pandas times), lat, lon (degrees), depth (km)
    max_dt_s   : the largest origin-time difference of a pair (s)
    returns    : DataFrame with ref's index and columns match, dt_s, dist_km, ddepth_km (other minus ref)
    """
    # Step 1: refuse times without a time zone; then put both tables on one clock (UTC).
    # Step 2: for every ref row, find the other row closest in time; keep it only within max_dt_s.
    # Step 3: for a pair, compute the time difference, the distance along the surface and the depth difference.
    # YOUR CODE HERE
    raise NotImplementedError("M32-T3: write your code here")

In [ ]:
st.check("M32-T3", func=match_locations, slug=PQ_SLUG)

PocketQuake's HYPOINVERSE (kim1983) against KMA and against your two M31 locations. `dmin_km`, the distance to the
nearest station with a weighted reading, comes from your archive table of task 2.

In [ ]:
dmin83 = phases["kim1983"][phases["kim1983"]["weight"] > 0.1].groupby("event")["distance_km"].min()
view = hyp83.assign(dmin_km=hyp83["id"].map(dmin83))
print("HYPOINVERSE, kim1983:")
print(view[["id", "time", "lat", "lon", "depth", "num", "gap", "dmin_km", "rms", "erh", "erz", "qual"]]
      .to_string(index=False))
vs_kma = match_locations(hyp83, kma_tab, MAX_DT_S)
vs_kma.insert(0, "kma_event", [kma_tab.loc[m, "event_id"] if m is not None else None for m in vs_kma["match"]])
print("\nKMA minus HYPOINVERSE (kim1983):")
print(vs_kma.drop(columns="match").round(3).to_string())
vs_m31 = match_locations(m31_tab, hyp83, MAX_DT_S)
vs_m31.insert(0, "location", m31_tab["location"])
print("\nPocketQuake's HYPOINVERSE (kim1983) minus your M31 locations of the mainshock:")
print(vs_m31.drop(columns="match").round(3).to_string())
lat_c = float(hyp83["lat"].mean())
print(f"\nKMA's rounding at {lat_c:.1f} N: 0.01 degree is {0.01 * 111.19:.2f} km north-south and "
      f"{0.01 * 111.19 * np.cos(np.radians(lat_c)):.2f} km east-west")
vs_kma.to_csv(Path(RESULT_DIR) / "M32_hypoinverse_vs_kma.csv")

A map (PyGMT, as in M02): KMA's epicenters (gray squares; several fall on the same rounded point), PocketQuake's
HYPOINVERSE (red circles) and your two M31 locations of the mainshock (blue star: Geiger; blue diamond: HYPOINVERSE
on your picks).

In [ ]:
import pygmt

region = [128.07, 128.13, 36.485, 36.525]
with st.plot.gmt_style(FONT_TITLE="12p,Helvetica"):
    fig_map = pygmt.Figure()
    fig_map.basemap(region=region, projection="M10c", frame=["af", "+tSangju earthquakes located three ways"])
    fig_map.plot(x=kma_tab.loc[vs_kma["match"].dropna(), "lon"], y=kma_tab.loc[vs_kma["match"].dropna(), "lat"],
                 style="s0.35c", fill="gray70", pen="0.5p,gray30", label="KMA catalog+S0.25c")
    fig_map.plot(x=hyp83["lon"], y=hyp83["lat"], style="c0.2c", fill="red3", pen="0.4p,black",
                 label="HYPOINVERSE kim1983+S0.2c")
    for name, style, text in (("your geiger", "a0.4c", "your Geiger location (M31)+S0.3c"),
                              ("HYPOINVERSE on your picks", "d0.3c", "HYPOINVERSE on your picks (M31)+S0.25c")):
        rows = m31_tab[m31_tab["location"] == name]
        if len(rows):
            fig_map.plot(x=rows["lon"], y=rows["lat"], style=style, fill="dodgerblue", pen="0.5p,black", label=text)
    fig_map.legend(position="jTL+o0.2c", box="+gwhite+p0.5p")
    fig_map.basemap(map_scale=st.plot.scale_bar_spec(region, km=1))
fig_map.savefig(Path(FIGURE_DIR) / "three_locations.png", dpi=150)
fig_map.show()

**Explain (3-5 sentences):** How far apart are HYPOINVERSE's and KMA's epicenters compared with KMA's rounding, and
which earthquake's **depth** (glossary `focal_depth`) differs most? Do its `erz`, `num`, `gap` and `dmin_km` tell
you how well HYPOINVERSE fixes that depth? How far are your two M31 locations from PocketQuake's mainshock, and
what does the difference between the two HYPOINVERSE runs (same program, model and commands) tell you about picks
and stations?

## Part 5 · An unchanged rerun and a second velocity model (about 75 min, task 4)

Two habits for a production tool, as in M21, M30 and M31: first an unchanged rerun (is the result **deterministic**,
glossary `deterministic_computation`, byte for byte?), then one thing changed. Here the change is the **velocity
model** (glossary `velocity_model`): the same picks, phase file and settings, located once with kim1983 and once with
kim2011. PocketQuake's stage writes both in one run, so changing the model means reading the other folder.

The cells copy the folder `1.HypoInv` after your first run (`results/hypinv/first`), rerun the location stage
alone (about a second), and copy the folder again (`results/hypinv/rerun`): a run overwrites the folder, so only a
copy keeps the earlier result.

### M32-T4 · An unchanged rerun and a second velocity model · core · about 75 min

**Goal:** write `differing_lines(path_a, path_b)`, use it to compare your first run with an unchanged rerun, and
measure how the locations change from kim1983 to kim2011.

**Given:** the copy and rerun cells below, and your `match_locations` of task 3.

**Contract (the function):**
- Input: `path_a`, `path_b` (`str` or `pathlib.Path`) of two text files.
- Output: a list of tuples `(line_number, line_a, line_b)`, one for every line number at which the two files
  differ, in order; line numbers count from 1, and lines are given without their end-of-line characters. A line that
  exists in only one file appears with None for the other. Two files that hold the same lines give `[]`.
- Example (made up): files with the lines `a`, `b`, `c` and `a`, `x`, `c`, `d` give `[(2, "b", "x"), (4, None,
  "d")]`.

**Contract (the report):** `rerun_report = {"differ": differing_lines, "slug": PQ_SLUG, "first": <the first copy's
folder>, "rerun": <the rerun copy's folder>, "fingerprints": {"first": <fingerprints of the first copy>, "rerun":
<fingerprints of the rerun copy>}, "model_change": <your match_locations with ref = the kim1983 summary and other =
the kim2011 summary of the first copy>}`, where the fingerprints of a copy are a dict from the path of every file
inside it, relative to the copy and written with / (for example `"kim1983/<Region>.sum"`), to the SHA-256 of that
file's bytes (glossary `checksum`; 64 hex letters, as `hexdigest()` writes them).

**Done when** `st.check("M32-T4", answer=rerun_report)` passes: the check runs your function on 8 made-up pairs of
files, then reads your two copies: it recomputes every fingerprint you give from the files, their .sum and .arc
files must be among the products your runs logged, the rerun must be a run of its own, and `model_change` must fit
the two summary files. It never prints the values.

**Stuck?** `st.hint("M32-T4", 1)`

In [ ]:
# Before you run: in M31 a rerun of hyp1.40 on your picks left one file different. PocketQuake's rerun also rewrites
# its phase and station files: will every file of the folder be byte-identical, and if not, which and why? Then,
# kim1983 against kim2011 with the same picks: will the epicenters or the depths move more, will the depths go up or
# down, and will the RMS residual change?
st.predict("M32-T4", """

""")

In [ ]:
def differing_lines(path_a, path_b):
    """The lines at which two text files differ: a list of (line_number from 1, line_a, line_b).

    path_a, path_b : str or pathlib.Path of the two files
    returns        : list of tuples; None for a line that only one file has; [] for files with the same lines
    """
    # Step 1: read both files as lines without their end-of-line characters.
    # Step 2: walk through the line numbers of the longer file; a missing line is None.
    # Step 3: keep the numbers at which the two lines differ, counting from 1.
    # YOUR CODE HERE
    raise NotImplementedError("M32-T4: write your code here")

In [ ]:
COPIES = Path(RESULT_DIR) / "hypinv"


def keep_copy(label):
    """Copy the cluster's 1.HypoInv folder to results/hypinv/<label> (the next run overwrites the original)."""
    target = COPIES / label
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(HYP_DIR, target)
    print(f"copied {sum(1 for p in target.rglob('*') if p.is_file())} files to {target}")
    return target


first_dir = keep_copy("first")
st.pq.run(PQ_SLUG, "hypoinverse", "hypoinverse", cores=PQ_CORES, mem_gb=PQ_MEM_GB)     # dry run

In [ ]:
assert st.pq.run(PQ_SLUG, "hypoinverse", "hypoinverse", cores=PQ_CORES, mem_gb=PQ_MEM_GB, apply=True) == 0
rerun_dir = keep_copy("rerun")

In [ ]:
# Fill in rerun_report: fingerprint every file of both copies with hashlib (each keyed by its path relative to its
# copy), and build model_change from the two summary files of the first copy (st.pq.read_sum takes a path to a .sum
# file; sum_table makes its times aware).
# YOUR CODE HERE
raise NotImplementedError("M32-T4: write your code here")
print({k: v for k, v in rerun_report.items() if k in ("slug", "first", "rerun")})

In [ ]:
st.check("M32-T4", answer=rerun_report)

Which files differ between the two copies (from your fingerprints), what differs in them, and how the locations
move from kim1983 to kim2011 (kim2011 minus kim1983):

In [ ]:
prints = rerun_report["fingerprints"]
differ_files = [rel for rel, digest in prints["first"].items() if prints["rerun"].get(rel) != digest]
print(f"files with the same SHA-256 in both copies: {len(prints['first']) - len(differ_files)} of "
      f"{len(prints['first'])}; files that differ: {differ_files}")
for rel in differ_files:
    if (rerun_dir / rel).is_file():
        for number, a, b in differing_lines(first_dir / rel, rerun_dir / rel):
            print(f"{rel}, line {number}:\n  first: {a}\n  rerun: {b}")
both = {m: sum_table(st.pq.read_sum(sorted((first_dir / m).glob("*.sum"))[0])) for m in MODELS}
change = rerun_report["model_change"].copy()
change.insert(0, "id", both["kim1983"]["id"].to_numpy())
for col in ("depth", "rms", "erz", "qual"):
    change[f"{col}_83"] = both["kim1983"][col].to_numpy()
    change[f"{col}_11"] = both["kim2011"][col].to_numpy()
print("\nkim2011 minus kim1983:")
print(change.drop(columns="match").round(3).to_string(index=False))
for name in MODELS:
    layers = st.data.velocity_model(name)
    print(f"\n{name}:", "; ".join(f"from {r.top_km:g} km: Vp {r.vp:.2f}, Vs {r.vs:.2f} km/s"
                                  for r in layers.itertuples()))

fig, ax = plt.subplots(figsize=(7.0, 3.6))
x = np.arange(len(change))
ax.plot(x, change["depth_83"], "o", color="tab:red", label="kim1983")
ax.plot(x, change["depth_11"], "s", color="tab:blue", label="kim2011")
ax.set_xticks(x, [str(i) for i in change["id"]], rotation=30)
ax.invert_yaxis()
ax.set_xlabel("Earthquake (PocketQuake id)")
ax.set_ylabel("Depth (km)")
fig.legend(loc="upper center", ncol=2, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(Path(FIGURE_DIR) / "depth_two_models.png", dpi=150)
plt.show()

**Explain (3-5 sentences):** Was the rerun identical, and if one file differed, in what and does it matter? From
kim1983 to kim2011, how far did the epicenters and the depths move, and how did the RMS and the quality change?
Explain the depth changes from the two models' layers (the speeds above the sources and the depth of the boundaries
below and above them). Which model would you use for these earthquakes, and why is a smaller RMS not the whole
argument?

## Part 6 · Early and late stations (about 75 min, task 5)

A **station delay** (glossary `station_delay`) is a constant time that a locator adds to the predicted travel time
at one station. HYPOINVERSE can hold one for P and one for S at every station (the P and S delay columns of the
archive file, 0 in your run). It stands for what a one-dimensional model leaves out near a station: slow sediments
or fast rock under it, or its height above the model's top surface, which HYPOINVERSE's layer model ignores (Klein
2002 suggests simulating station elevations with delays). With many earthquakes, a station whose residuals are
consistently positive (late) or negative (early) is a candidate for a delay. Programs such as VELEST find delays
together with a one-dimensional model from many earthquakes all around the stations (Kissling et al. 1994).

"Consistently" needs a measure. With n residuals of mean m and sample standard deviation s, the **standard error**
(glossary `standard_error`) of the mean is s/√n; a mean more than two standard errors from zero is unlikely to be
chance.

### M32-T5 · Early and late stations · core · about 75 min

**Goal:** write `station_terms(phases, min_events)`, which summarizes the residuals of every station and phase, and
use it on both models.

**Given:** your `read_arc` of task 2 and its tables `phases["kim1983"]` and `phases["kim2011"]`.

**Contract:**
- Input: `phases`, a DataFrame like those of `read_arc` (at least `event`, `station`, `phase`, `residual_s` and
  `weight`, any index and row order); `min_events` (int).
- Use only the readings that HYPOINVERSE used: weight above 0.1 (the rule of the summary's `num`).
- Output: a pandas DataFrame with one row per station and phase that has at least `min_events` such readings, and
  the columns `station`, `phase`, `n` (the number of readings), `mean_s` (their mean residual), `se_s` (their
  standard error: the sample standard deviation, with n − 1, divided by √n) and `label`: `"late"` when
  `mean_s − 2 se_s > 0`, `"early"` when `mean_s + 2 se_s < 0`, else `""`. Rows sorted by station, then phase.
- Example (made up): station XX.QQA1 with the P residuals 0.20, 0.30 and 0.25 s (weights 0.9, 1.2, 0.8) and one
  of 8.0 s with weight 0 gives, with `min_events` 3: n 3, mean_s 0.25, se_s 0.0289 and label "late".

**Done when** `st.check("M32-T5", func=station_terms, reader=read_arc, slug=PQ_SLUG)` passes: 10 made-up tables
whose station means and standard errors are fixed by design (with readings of weight 0, 0.05 and exactly 0.10 that
must stay out), then your two archive files read with your `read_arc`: every row needs enough readings, no left-out
reading may reach a mean, and the labels must follow the rule (only properties; no values are printed).

**Stuck?** `st.hint("M32-T5", 1)`

In [ ]:
# Before you code: under kim1983, do you expect the stations that stand out to be a few scattered ones, or a pattern
# shared by many stations (for example most P late, or most S early)? What would such a shared pattern say about the
# model rather than about the stations, and what do you expect under kim2011?
st.predict("M32-T5", """

""")

In [ ]:
def station_terms(phases, min_events):
    """Mean residual, standard error and label of every station and phase with at least min_events used readings.

    phases     : DataFrame with event, station, phase, residual_s, weight (as read_arc gives)
    min_events : the smallest number of used readings (weight above 0.1) for a row
    returns    : DataFrame with station, phase, n, mean_s, se_s, label ('late', 'early' or ''), sorted
    """
    # Step 1: keep the readings with weight above 0.1.
    # Step 2: per station and phase: the number, the mean and the sample standard deviation (n - 1) of the residuals.
    # Step 3: keep the groups with at least min_events readings; the standard error is the deviation over the root
    #         of n; label by the two-standard-error rule; sort.
    # YOUR CODE HERE
    raise NotImplementedError("M32-T5: write your code here")

In [ ]:
st.check("M32-T5", func=station_terms, reader=read_arc, slug=PQ_SLUG)

Your station terms in both models: how many are early or late, the largest under kim2011, and the kim2011 terms
against the height of each station (from PocketQuake's station table; the layer model puts every station at height
0).

In [ ]:
from scipy.stats import linregress

terms = {m: station_terms(phases[m], MIN_EVENTS) for m in MODELS}
for m in MODELS:
    t = terms[m]
    print(f"{m}: {len(t)} station-phases with at least {MIN_EVENTS} readings")
    print(pd.crosstab(t["phase"], t["label"].replace("", "neither")).to_string(), "\n")
big = terms["kim2011"].reindex(terms["kim2011"]["mean_s"].abs().sort_values(ascending=False).index).head(10)
print("kim2011, the ten largest station terms:")
print(big.round(3).to_string(index=False))
terms["kim2011"].to_csv(Path(RESULT_DIR) / "M32_station_terms_kim2011.csv", index=False)

stations = pd.read_csv(RUN_DIR / "station_table" / "used_stations_100km.csv")
height_km = pd.Series(stations["Elevation"].to_numpy() / 1000.0,
                      index=stations["Network"] + "." + stations["Code"])
top = st.data.velocity_model("kim2011").iloc[0]
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8), sharey=True)
for ax, phase, speed in zip(axes, ("P", "S"), (top.vp, top.vs)):
    t = terms["kim2011"][terms["kim2011"]["phase"] == phase]
    h = t["station"].map(height_km)
    fit = linregress(h, t["mean_s"])
    print(f"kim2011 {phase}: slope {fit.slope:.3f} +- {fit.stderr:.3f} s per km of height (r = {fit.rvalue:.2f}, "
          f"{len(t)} stations); a vertical ray in the top layer: {1 / speed:.3f} s per km")
    ax.errorbar(h, t["mean_s"], yerr=t["se_s"], fmt="o", ms=4, color="tab:blue", ecolor="0.6", label="Station term")
    hh = np.linspace(0, float(h.max()), 2)
    ax.plot(hh, fit.intercept + fit.slope * hh, color="tab:red", label="Least-squares line")
    ax.plot(hh, fit.intercept + hh / speed, "--", color="0.3", label="Vertical ray, top layer")
    ax.axhline(0, color="0.8", lw=0.8)
    ax.set_title(f"{phase} waves, kim2011", fontsize=10)
    ax.set_xlabel("Station height (km)")
axes[0].set_ylabel("Mean residual (s)")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(Path(FIGURE_DIR) / "station_terms_height.png", dpi=150)
plt.show()

**Explain (3-5 sentences):** Under kim1983, how many station-phases were early or late, and on which side for P and
for S? What does a pattern shared by most stations tell you about the model (think of the **Vp/Vs ratio**, glossary
`vp_vs_ratio`, and M30's Wadati diagram) rather than about the stations? Under kim2011, how do the terms relate to
the station heights, compared with a vertical ray in the top layer? Why would you hesitate to put these terms into
HYPOINVERSE as station delays, from six earthquakes of one cluster?

## What you made, and what comes next

- `results/hypinv/first` and `results/hypinv/rerun`: your two location folders; `work/.sgtl/runs.jsonl` logs the
  runs with the fingerprints of their products.
- `results/M32_hypoinverse_vs_kma.csv`: KMA minus HYPOINVERSE (kim1983); `results/M32_station_terms_kim2011.csv`:
  your station terms under kim2011.
- `figures/three_locations.png`, `figures/depth_two_models.png` and `figures/station_terms_height.png`.

Module M41 relocates these earthquakes relative to each other with HypoDD, starting from the kim1983 archive file of
your cluster (PocketQuake's `ph2dt` stage reads it). Your cluster `PQ_SLUG` stays registered.

The station coordinates in these files come from STP's station tables in the SAC headers; the course's station
metadata (StationMetadata 1.0.2 by Byeong-Seok Ahn, Pukyong National University; reference key `ahn2024`) is not
needed here, because HYPOINVERSE uses arrival times only. The KMA catalog is used under the Korea Open Government
License Type 1 (attribution).

Your progress in this module (from `work/.sgtl/progress.jsonl`):

In [ ]:
st.progress("M32")

### Going further (optional, not checked)

1. Plot the kim1983 residuals of task 5 against distance for P and S. Does the head wave along the 15-km boundary
   (M30-T2) show up as a change of slope?
2. Repeat task 5 with `MIN_EVENTS = 3` and `5`. Which labels are stable?
3. Rerun the location with the M21 threshold of 0.5 (set `PNPLUS_MIN_PROB` and run `picking` to `hypoinverse` as in
   M21-T4). How much do `num`, the depths and the RMS change, and why so little?